### Задание
1) Загрузите модель эмбеддингов для русского языка и проанализируйте, какой препроцессинг требуется для успешной работы с ней.  
2) Подготовьте собственный датасет на русском языке, обучите **word2vec** и сравните ближайших соседей с готовыми эмбеддингами.  
3) Постройте индекс по вашему датасету. Если ваши данные — одно длинное произведение, разбейте его на части (абзацы или фрагменты по 100–500 слов). Проверьте работу поиска по вашему индексу на нескольких запросах.

### Что сдавать
1) Список выбранных слов и их ближайшие соседи (готовая русская модель; задание 1).  
2) 5–10 примеров ближайших слов + 2–3 наблюдения (word2vec на ваших данных; задание 2).  
3) Для retrieval: 3–5 запросов и топ-5 документов + короткий вывод (задание 3).


## 0) Установка и импорты

Если `gensim` или `faiss` не установлены в вашей среде, раскомментируйте `pip install` ниже.

In [245]:
!pip -q install gensim
!pip -q install faiss-cpu  # для CPU-индекса (Windows иногда требует перезапуск kernel)

import re
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec


## 1) Данные:
# Датасет с категориями: https://huggingface.co/datasets/data-silence/rus_news_classifier

Категория: 1 - conflicts, 8 - society


In [246]:
# --- Загрузка данных ---
from datasets import load_dataset

ds = load_dataset("data-silence/rus_news_classifier")

#  1 - conflicts, 8 - society
text_conflicts = [i["news"] for i in ds["train"] if i["labels"] == 1]
text_society  = [i["news"] for i in ds["train"] if i["labels"] == 8]

X_text = np.array(text_conflicts + text_society, dtype=object)
y = np.array([0]*len(text_conflicts) + [1]*len(text_society))

print("Всего документов:", len(X_text))
print("Баланс классов:", dict(zip(*np.unique(y, return_counts=True))))

Всего документов: 11214
Баланс классов: {np.int64(0): np.int64(6277), np.int64(1): np.int64(4937)}


In [247]:
# 5 - health
text_health = [i["news"] for i in ds["train"] if i["labels"] == 5]

Загрузим корпус и подготовим простую токенизацию.

In [248]:
TOKEN_RE = re.compile(r"[А-Яа-яЁё]+(?:'[А-Яа-яЁё]+)?")
def tokenize(text):
    # Очень простая токенизация для учебных целей
    # 1) в нижний регистр
    # 2) оставляем только латиницу
    # 3) режем по пробелам
    text = text.lower()
    tokens = re.findall(TOKEN_RE, text)
    return tokens

tokenized_texts = [tokenize(t) for t in text_conflicts]
print("Пример токенов:", tokenized_texts[0][:25])


Пример токенов: ['житель', 'москвы', 'сходил', 'на', 'сеанс', 'эротического', 'массажа', 'после', 'которого', 'умер', 'об', 'этом', 'сообщает', 'канал', 'по', 'информации', 'издания', 'летний', 'москвич', 'заказывал', 'сеанс', 'массажа', 'с', 'последующими', 'интимными']


Сделаем лемматизацию для русского языка и удалим стоп-слова

In [249]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords
ru_stop = stopwords.words("russian")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [250]:
pip install pymorphy3

In [251]:
import pymorphy3
morph = pymorphy3.MorphAnalyzer()

In [252]:
from functools import lru_cache

In [253]:
@lru_cache(maxsize=200_000)
def lemma_cached(w: str) -> str:
    return morph.parse(w)[0].normal_form

def simple_tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(text.lower())

def tokenize_lemmas(text: str) -> list[str]:
    token = []
    words = simple_tokenize(text)
    for w in words:
        if w in ru_stop:
            continue
        lemma = lemma_cached(w)
        if lemma in ru_stop:
            continue
        token.append(lemma)
    return token

In [254]:
tokens_conflicts = [tokenize_lemmas(t) for t in text_conflicts]
print("Пример токенов:", tokens_conflicts[0][:25])

Пример токенов: ['житель', 'москва', 'сходить', 'сеанс', 'эротический', 'массаж', 'который', 'умереть', 'сообщать', 'канал', 'информация', 'издание', 'летний', 'москвич', 'заказывать', 'сеанс', 'массаж', 'последующий', 'интимный', 'услуга', 'квартира', 'бульвар', 'ян', 'райнис', 'некоторый']


In [255]:
tokens_society = [tokenize_lemmas(t) for t in text_society]
print("Пример токенов:", tokens_society[0][:25])

Пример токенов: ['собор', 'святой', 'марк', 'построить', 'век', 'венеция', 'пострадать', 'результат', 'наводнение', 'миллион', 'доллар', 'сообщать', 'издание', 'наводнение', 'ноябрь', 'собор', 'закрыть', 'неделя', 'реконструкция', 'пояснить', 'сотрудник', 'строение', 'внешний', 'повреждение', 'удаться']


In [256]:
tokens_health = [tokenize_lemmas(t) for t in text_health]
print("Пример токенов:", tokens_health[0][:25])

Пример токенов: ['человек', 'проживать', 'рядом', 'дорога', 'частый', 'страдать', 'высокий', 'кровяной', 'давление', 'опасный', 'последствие', 'шумовой', 'загрязнение', 'назвать', 'профессор', 'медицина', 'оксфордский', 'университет', 'казем', 'рахимя', 'комментарий', 'ведущий', 'автор', 'исследование', 'казем']


## 2) Задание 1: готовые эмбеддинги и ближайшие соседи

Попробуем загрузить готовые вектора через `gensim.downloader`.

Рекомендуемые варианты:
- `glove-wiki-gigaword-100` (обычно быстрее скачивается)
- `word2vec-google-news-300` (очень большой)

Если скачивание недоступно — пропустите к заданию 2 (обучение своих эмбеддингов) и используйте их вместо готовых.

In [257]:
#Посмотреть, какие модели доступны:
print(list(api.info()['models'].keys())[:25])

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [258]:
MODEL_NAME = "word2vec-ruscorpora-300"

try:
    wv = api.load(MODEL_NAME)  # KeyedVectors
    print("Загружено:", MODEL_NAME)
    print("Словарь модели:", len(wv))
    print("Размерность:", wv.vector_size)
except Exception as e:
    wv = None
    print("Не удалось загрузить готовые эмбеддинги:", repr(e))
    print("Продолжайте с обучением собственных эмбеддингов ниже (Задание 2).")


Загружено: word2vec-ruscorpora-300
Словарь модели: 184973
Размерность: 300


In [259]:
print(wv.index_to_key[:25])

['весь_DET', 'человек_NOUN', 'мочь_VERB', 'год_NOUN', 'сказать_VERB', 'время_NOUN', 'говорить_VERB', 'становиться_VERB', 'знать_VERB', 'самый_DET', 'дело_NOUN', 'день_NOUN', 'жизнь_NOUN', 'рука_NOUN', 'очень_ADV', 'первый_ADJ', 'давать_VERB', 'новый_ADJ', 'слово_NOUN', 'иметь_VERB', 'большой_ADJ', 'идти_VERB', 'глаз_NOUN', 'место_NOUN', 'лицо_NOUN']


## Выбираем слова из корпуса и смотрим соседей

Выберите 10–20 слов (желательно тематических) и посмотрите ближайшие соседи.

**Задача:**
- Выберите по 3–5 слов из разных доменов (например, религия/спорт/компьютеры/медицина).
- Для каждого слова распечатайте 10 ближайших соседей.
- Сделайте 3–5 наблюдений: где соседи хорошие, где странные, почему так могло получиться.

In [260]:
# слова по доменам
words_domens = {
    "конфликты": tokens_conflicts[0][:5],
    "общество": tokens_society[0][:5],
    "здоровье": tokens_health[0][:5]
}
words_domens

{'конфликты': ['житель', 'москва', 'сходить', 'сеанс', 'эротический'],
 'общество': ['собор', 'святой', 'марк', 'построить', 'век'],
 'здоровье': ['человек', 'проживать', 'рядом', 'дорога', 'частый']}

In [261]:
def print_neighbors(model, words, topn=10):
  tags = ["_NOUN", "_VERB", "_ADJ", "_ADV", "_DET"]
  for word in words:
    found = False
    for tag in tags:
        candidate = word + tag
        if candidate in model.key_to_index:
          found = True
          print(f"\n=== {candidate} ===")
          for neighbor, score in model.most_similar(candidate, topn):
            print(f"{neighbor:20} {score:.4f}")
    if not found:
      print(f"\n=== {word} ===")
      print("Слова нет в словаре модели")

In [262]:
print_neighbors(wv, words_domens["конфликты"])


=== житель_NOUN ===
обитатель_NOUN       0.4060
население_NOUN       0.4042
горожанин_NOUN       0.3793
чукоча_NOUN          0.3468
багамойо_NOUN        0.3401
абориген_NOUN        0.3395
необитаемость_NOUN   0.3368
окрестный_ADJ        0.3363
близлежащий_ADJ      0.3339
ахиол_NOUN           0.3277

=== москва_NOUN ===
калуга_NOUN          0.4312
тверь_NOUN           0.4268
ярославль_NOUN       0.4116
казань_NOUN          0.4097
рязань_NOUN          0.3966
московский_ADJ       0.3961
столица_NOUN         0.3945
коломна_NOUN         0.3920
ленинград_NOUN       0.3840
тула_NOUN            0.3798

=== сходить_VERB ===
сбегать_VERB         0.3359
спускаться_VERB      0.3061
тронуться_VERB       0.2688
слетать_VERB         0.2502
ступать_VERB         0.2487
пойти_VERB           0.2443
взбегать_VERB        0.2440
побежать_VERB        0.2419
спрыгивать_VERB      0.2377
проскользить_VERB    0.2369

=== сеанс_NOUN ===
эриксоновский_ADJ    0.3349
киносеанс_NOUN       0.2868
стереокино_NOUN     

Наблюдение: Модель эмбеддингов хорошо работает со словами, принадлежащими к одному и тому же семантическому полю. Наибольшее сходство (0,38-0,44) наблюдается у конкретных существительных, в то время как глаголы и многозначные существительные имеют меньшее сходство (0,24-0,33).

In [263]:
print_neighbors(wv, words_domens["общество"])


=== собор_NOUN ===
церковь_NOUN         0.4422
соборный_ADJ         0.4360
кафедральный::собор_NOUN 0.4179
пятиглавый_ADJ       0.4161
храм_NOUN            0.4146
богоматерь_NOUN      0.4142
успенский::собор_NOUN 0.4072
дворищенский_ADJ     0.3841
никитник_NOUN        0.3833
придельный_ADJ       0.3757

=== святой_NOUN ===
святой_ADJ           0.5046
святый_ADJ           0.4810
св_NOUN              0.4624
апостол_NOUN         0.4173
богоматерь_NOUN      0.4111
священномученик_NOUN 0.4027
свв_NOUN             0.3929
иоанн::златоустый_NOUN 0.3910
святитель_NOUN       0.3828
сошествие_NOUN       0.3807

=== святой_ADJ ===
святой_NOUN          0.5458
св_NOUN              0.4718
святый_ADJ           0.4414
святитель_NOUN       0.4350
целитель::пантелеймон_NOUN 0.4225
священномученик_NOUN 0.4185
мироточивый_ADJ      0.4174
богоматерь_NOUN      0.4153
святая_NOUN          0.4147
водосвятный::молебен_NOUN 0.4139

=== марк_NOUN ===
бершадский_NOUN      0.2914
джакобс_NOUN         0.2844
матфей

Наблюдение: слова из предложения по научной категории были мало похожи на эту категории и ближашие соседи тоже не асоцируются с научными  

In [264]:
print_neighbors(wv, words_domens["здоровье"])


=== человек_NOUN ===
мужчина_NOUN         0.3356
человеческий_ADJ     0.3096
женщина_NOUN         0.3073
неприрученный_ADJ    0.2596
индивидуум_NOUN      0.2565
нубиец_NOUN          0.2536
манкурт_NOUN         0.2532
обезъян_NOUN         0.2500
обездвиженный_ADJ    0.2474
центровой::образованщина_NOUN 0.2461

=== проживать_VERB ===
жить_VERB            0.4108
поселяться_VERB      0.3491
скитаться_VERB       0.3214
пожить_VERB          0.3061
переселяться_VERB    0.2970
расселять_VERB       0.2868
проживание_NOUN      0.2739
проскитаться_VERB    0.2737
воронино_NOUN        0.2699
доживать_VERB        0.2688

=== рядом_ADV ===
неподалеку_ADV       0.4253
возле_ADP            0.4138
рядышком_ADV         0.4105
позади_ADP           0.3969
недалеко_ADV         0.3710
невдалеке_ADV        0.3680
поодаль_ADV          0.3540
подле_ADP            0.3516
справа_ADV           0.3497
слева_ADV            0.3464

=== дорога_NOUN ===
шоссе_NOUN           0.4491
проселок_NOUN        0.4178
просек_NO

Наблюдение: на категории здоровье ближашие соседи на словах человек, проживать похожи по семантике. но тоже есть странные слова: рядом, дорога, частый



## 3)  Задание 2: обучаем Word2Vec на тексте о здоровье

**Задача:**
- Обучите модель.
- Проверьте ближайшие слова к названиям команд: `rangers`, `bruins`, `canadiens`, `flyers`, `leafs`, `oilers` и т.п.
- Сравните с готовыми эмбеддингами (если они доступны).

In [265]:
w2v_health = Word2Vec(
    sentences=tokens_health,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,              # 1 = skip-gram, 0 = CBOW
    negative=10,
    epochs=10,
    workers=4
)

w2v_health = w2v_health.wv
print("Словарь путешествие модели:", len(w2v_health))
print("Размерность:", w2v_health.vector_size)


Словарь путешествие модели: 13746
Размерность: 100


In [266]:
teams = ["больница", "врач", "лечение", "здоровье", "болезнь", "пациент", "лекарство", "хирургия", "терапия", "медицина"]
for t in teams:
    if t in w2v_health:
        print("\n", t, "->", [w for w, _ in w2v_health.most_similar(t, topn=10)])
    else:
        print("\n", t, "нет в словаре (проверь min_count или орфографию)")



 больница -> ['отделение', 'йорк', 'нью', 'реанимационный', 'мариинский', 'госпиталь', 'массачусетский', 'святой', 'госпитализировать', 'бристоль']

 врач -> ['евдокименко', 'токсиколог', 'парамонов', 'ужевко', 'хаас', 'фадейкин', 'боголюб', 'жовтан', 'артемьев', 'ревматолог']

 лечение -> ['терапия', 'медикаментозный', 'немедикаментозный', 'терапевтический', 'микрополяризация', 'рецидивировать', 'лучевой', 'борьба', 'малоинвазивный', 'дисульфира']

 здоровье -> ['пагубно', 'самочувствие', 'психика', 'суварна', 'безалкогольный', 'сказываться', 'ментальный', 'значиться', 'отзываться', 'вернер']

 болезнь -> ['альцгеймер', 'паркинсон', 'заболевание', 'крона', 'эмфизема', 'порок', 'ибс', 'гэрба', 'корсаков', 'нейродегенерация']

 пациент -> ['больной', 'человек', 'эшт', 'ревмокардит', 'регрессионный', 'прополис', 'пациентка', 'тимэктомия', 'госпитализировать', 'миелома']

 лекарство -> ['препарат', 'лекарственный', 'статин', 'антибиотик', 'нестероидный', 'парацетамол', 'антигистаминный',

## 4) Задание 3: индекс документов как матрица (mean Word2Vec)

Построим эмбеддинг документа как **среднее** эмбеддингов слов.

Два варианта:
1. Просто среднее.
2. **TF–IDF взвешенное** среднее (обычно лучше).

**Задача:**
- Постройте матрицу `D` размера (num_docs × dim).
- Напишите 3–5 запросов и посмотрите топ-5 ближайших документов.
- Оцените глазами: насколько выдача “в тему”.

In [267]:
X_train, X_test = train_test_split(text_health, test_size=0.2, random_state=42)
docs = X_test[:1000]  # ограничим

docs_tok = [tokenize_lemmas(t) for t in docs]
wv_use = w2v_health

DIM = wv_use.vector_size

def doc_vector_mean(tokens, wv_use):
    vecs = [wv_use[w] for w in tokens if w in wv_use]
    if not vecs:
        return np.zeros(DIM, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

# TF–IDF для взвешивания
vectorizer = TfidfVectorizer(tokenizer=tokenize_lemmas, lowercase=True, min_df=2, max_df=0.9)
tfidf = vectorizer.fit_transform(docs)  # (num_docs, vocab)
vocab = vectorizer.vocabulary_  # word -> col

def doc_vector_tfidf(tokens, wv_use, tfidf_row, vocab):
    # tfidf_row: sparse row (1, vocab_size)
    weights = {}
    # берём только слова из tokens, которые есть в vocab
    for w in tokens:
        j = vocab.get(w, None)
        if j is not None:
            weights[w] = tfidf_row[0, j]
    # соберём взвешенную сумму
    num = np.zeros(DIM, dtype=np.float32)
    den = 0.0
    for w, a in weights.items():
        if w in wv_use and a > 0:
            num += (a * wv_use[w]).astype(np.float32)
            den += float(a)
    if den == 0.0:
        return doc_vector_mean(tokens, wv_use)
    return (num / den).astype(np.float32)

# Матрица эмбеддингов документов
D_mean = np.vstack([doc_vector_mean(t, wv_use) for t in docs_tok])
D_tfidf = np.vstack([doc_vector_tfidf(docs_tok[i], wv_use, tfidf[i], vocab) for i in range(len(docs_tok))])

print("D_mean shape:", D_mean.shape, "D_tfidf shape:", D_tfidf.shape)


# Нормируем матрицы заранее для cosine (ускоряет поиск)
D_mean_n = D_mean / (np.linalg.norm(D_mean, axis=1, keepdims=True) + 1e-9)
D_tfidf_n = D_tfidf / (np.linalg.norm(D_tfidf, axis=1, keepdims=True) + 1e-9)
print('D_mean_n shape:', D_mean_n.shape, 'D_tfidf_n shape:', D_tfidf_n.shape)

D_mean shape: (987, 100) D_tfidf shape: (987, 100)
D_mean_n shape: (987, 100) D_tfidf_n shape: (987, 100)


In [268]:
def normalize_rows(M: np.ndarray) -> np.ndarray:
    M = M.astype(np.float32, copy=False)
    norms = np.linalg.norm(M, axis=1, keepdims=True) + 1e-9
    return M / norms

def cosine_topk_pre_norm(query_vec: np.ndarray, Dnrm: np.ndarray, k: int = 5):
    """Cosine top-k, where Dnrm already has unit-norm rows."""
    q = query_vec.astype(np.float32)
    q = q / (np.linalg.norm(q) + 1e-9)
    scores = Dnrm @ q
    idx = np.argsort(-scores)[:k]
    return idx, scores[idx]


In [269]:
# 5 запросов + вызовы: TF–IDF baseline + dense cosine

import re
import numpy as np

queries = [
    "Врач назначил пациенту курс лечения и выписал рецепт на лекарства.",
    "Здоровье человека зависит от образа жизни и своевременной профилактики болезней.",
    "В больнице провели сложную хирургическую операцию под общим наркозом.",
    "Пациент пожаловался на симптомы гриппа: высокая температура и кашель.",
    "Медицина XXI века использует передовые технологии для диагностики заболеваний."
]
K = 5

for q in queries:
    print("=" * 90)
    print("Query:", q)

    # (1) TF–IDF baseline: scores = tfidf @ q_tfidf^T
    q_tfidf = vectorizer.transform([q])                    # (1, |V|)
    scores_tfidf = (tfidf @ q_tfidf.T).toarray().ravel()   # (N,)
    idx_tfidf = np.argsort(-scores_tfidf)[:K]

    print("\nTF–IDF top-k:")
    for r, i in enumerate(idx_tfidf, 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={scores_tfidf[i]:.4f} | {snippet}...")

    # (2) Dense cosine over pre-normalized doc matrix D_tfidf_n
    tok = tokenize_lemmas(q)
    qv = doc_vector_tfidf(tok, wv_use, q_tfidf, vocab)     # (dim,)
    idx_dense, sc_dense = cosine_topk_pre_norm(qv, D_tfidf_n, k=K)

    print("\nDense (cosine) top-k:")
    for r, (i, s) in enumerate(zip(idx_dense, sc_dense), 1):
        snippet = re.sub(r"\s+", " ", docs[i])[:200]
        print(f"{r:>2}. score={float(s):.4f} | {snippet}...")

Query: Врач назначил пациенту курс лечения и выписал рецепт на лекарства.

TF–IDF top-k:
 1. score=0.1591 | Клиническое исследование специалистов из США показало, что антибиотики можно применять в качестве эффективной терапии первой линии при лечении аппендицита. Новая статья ученых опубликована в The New E...
 2. score=0.1208 | Эффективность нового препарата российского производства для лечения рака с помощью света может на определенных стадиях доходить до 90-100 процентов. Об этом рассказал врач по рентгенэндоваскулярной ди...
 3. score=0.1166 | Норвежские ученые из Университетской больницы Акерсхус и Университета Осло поставили под сомнение терапию антибиотиками при госпитализации с ОРВИ. Выводы исследования приводились на Европейском конгре...
 4. score=0.1144 | Российские и польские ученые разработают новый препарат против рака легкого, не разрушающий здоровые клетки. Об этом сообщают «Известия». По словам специалистов, лекарство сможет эффективно лечить люд...
 5. score=0.1103 | 

## Выводы
По результатам поиска видно, что обе модели находят документы, связанные с темой здоровья и медицины. Однако TF-IDF лучше находит тексты с точными совпадениями слов из запроса, например «врач», «лечение», «симптомы», «операция», «технологии». В то же время Dense-модель (эмбеддинги) находит более семантически близкие документы, даже если в тексте нет точных слов из запроса, например новости про исследования, лекарства, пациентов или медицинские процедуры.

В целом Dense-поиск даёт более общий и смысловой результат, хорошо понимая контекст заболеваний и методов лечения, а TF-IDF — более точное совпадение по ключевым словам, что особенно эффективно при поиске конкретных симптомов или названий болезней.